# Business Pipeline Demo（D01–D11）

> Issue #10

演示完整主链：接收 → 结构化 → 清洗 → 衍生 → 计划 → 指标 → 六视角 → 结论建议 → 报告归档。
上游 blocked 时不会生成正式交付。


In [ ]:
from pathlib import Path
import pandas as pd
from wanglaoshi import BusinessPipeline, intake_file

demo = Path('outputs/business_demo')
demo.mkdir(parents=True, exist_ok=True)
src = demo / 'orders.csv'
pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6],
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06'],
    'amount': [10.5, 20.0, 7.2, 80.0, 15.0, 12.0],
    'channel': ['web', 'app', 'web', 'app', 'web', 'app'],
    'user_tier': ['A', 'B', 'A', 'A', 'B', 'B'],
}).to_csv(src, index=False)

result = intake_file(src, demo / 'archive', required_columns=['order_id', 'amount'], period='2024-Q1')
print(result.status)
print(result.data)


In [ ]:
pipe = BusinessPipeline(
    demo / 'archive_run',
    required_columns=['order_id', 'amount', 'channel'],
    date_col='order_date',
    value_col='amount',
    dim_cols=['channel'],
    segment_cols=['user_tier'],
)
record = pipe.run(src)
print(record.summary())

d10 = record.stages[-1]
print('D10', d10.status, d10.data.get('delivery_index', {}).get('deliverable'))
for p in d10.output_paths:
    print(' -', p)


In [ ]:
# 查看报告摘要
from pathlib import Path
report = Path(d10.output_paths[1]) if len(d10.output_paths) > 1 else None
candidates = list((demo / 'archive_run').glob('*/report.md'))
path = candidates[0] if candidates else report
print(path)
print(path.read_text(encoding='utf-8')[:1200] if path and path.exists() else 'no report')


In [ ]:
# 缺列 → warning；缺文件 → blocked 且下游 skipped（无正式交付）
warn = intake_file(src, demo / 'archive2', required_columns=['order_id', 'amount', 'missing_col'])
print('warn', warn.status, warn.warnings)

blocked = BusinessPipeline(demo / 'archive3').run(demo / 'no_such.csv')
print('blocked', blocked.summary())
